# Demo D: Interactive Capacity Planning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/demos/demo_d_capacity_calculator.ipynb)

**Goal:** Derive KV cache cost from model config, visualize how it scales,
and calculate exactly how many users fit on each GPU.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers', 'matplotlib'])

from transformers import AutoConfig
import matplotlib.pyplot as plt

### The Capacity Equation

How many concurrent users fit on a given GPU?

```
Available = GPU_VRAM - model_weights - overhead
Max users = Available / (KV_per_token x context_length)
```

Change parameters with dropdowns and sliders. Watch the memory bar fill in real-time.

### Capacity Planning and GPU Selection

Interactive deployment calculator.

In [2]:
import numpy as np  # Import dependency
import matplotlib.pyplot as plt  # Import dependency
import ipywidgets as widgets  # Import dependency
from IPython.display import display, HTML  # Import dependency

# Model specifications: params in billions, layer/head/dim counts
MODELS = {  # Assign value
    'Mistral-7B':  {'params_B': 7.24,  'layers': 32,  'heads': 32,  'kv_heads': 8, 'head_dim': 128},
    'Llama-70B':   {'params_B': 70.6,  'layers': 80,  'heads': 64,  'kv_heads': 8, 'head_dim': 128},
    'Llama-405B':  {'params_B': 405.0, 'layers': 126, 'heads': 128,'kv_heads': 8, 'head_dim': 128},
}

# GPU VRAM in GB
GPUS = {  # Assign value
    'T4': 16, 'A10G': 24, 'A100-80': 80, 'H100': 80, 'H200': 141,
}

# Bytes per parameter for each precision
PRECISIONS = {  # Assign value
    'FP16': 2, 'INT8': 1, 'INT4': 0.5,
}

# Approximate cost per hour (USD) for reference
GPU_COST = {  # Assign value
    'T4': 0.53, 'A10G': 1.21, 'A100-80': 3.67, 'H100': 4.25, 'H200': 12.50,
}

print('Models:', list(MODELS.keys()))  # Display output
print('GPUs:', list(GPUS.keys()))  # Display output
print('Precisions:', list(PRECISIONS.keys()))  # Display output

# --- Batch analysis (A100-80, Mistral-7B FP16) ---

# GPU and model parameters (A100 80GB)
bandwidth_gbs = 2039        # HBM bandwidth in GB/s
model_size_gb = 14.5        # Model weights in GB (7B params in fp16)
kv_per_token_mb = 2         # KV cache per token in MB

# Batch sizes to evaluate
batch_sizes = np.array([1, 2, 4, 8, 16, 32, 64, 128])  # NumPy operation

# --- Instance catalog ---

# GPU instance catalog: real cloud offerings
GPU_INSTANCES = {  # Assign value
    "A10G (24GB)": {"vram_gb": 24, "bandwidth_gbs": 600, "flops_tflops": 31.2, "cost_per_hr": 1.01},
    "L4 (24GB)": {"vram_gb": 24, "bandwidth_gbs": 300, "flops_tflops": 30.3, "cost_per_hr": 0.81},
    "A100 40GB": {"vram_gb": 40, "bandwidth_gbs": 1555, "flops_tflops": 77.9, "cost_per_hr": 3.67},
    "A100 80GB": {"vram_gb": 80, "bandwidth_gbs": 2039, "flops_tflops": 77.9, "cost_per_hr": 5.12},
    "H100 80GB": {"vram_gb": 80, "bandwidth_gbs": 3350, "flops_tflops": 267.6, "cost_per_hr": 8.10},
    "L40S (48GB)": {"vram_gb": 48, "bandwidth_gbs": 864, "flops_tflops": 91.6, "cost_per_hr": 2.40},
    "H200 (141GB)": {"vram_gb": 141, "bandwidth_gbs": 4800, "flops_tflops": 267.6, "cost_per_hr": 12.50},
    "MI300X (192GB)": {"vram_gb": 192, "bandwidth_gbs": 5300, "flops_tflops": 163.4, "cost_per_hr": 10.00},
}

# Bytes per parameter by precision
PRECISION_BYTES = {"FP16": 2, "INT8": 1, "INT4": 0.5}  # Assign value

# Model sizes in billions of parameters
MODEL_PARAMS = {"7B": 7, "13B": 13, "70B": 70, "405B": 405}  # Assign value


Models: ['Mistral-7B', 'Llama-70B', 'Llama-405B']
GPUs: ['T4', 'A10G', 'A100-80', 'H100', 'H200']
Precisions: ['FP16', 'INT8', 'INT4']


In [3]:
def capacity_calc(model_name, gpu_name, precision, tokens_per_conversation, num_users):
    """Show every calculation step for GPU memory capacity planning."""
    m = MODELS[model_name]  # Assign value
    gpu_vram = GPUS[gpu_name]  # Assign value
    bpp = PRECISIONS[precision]  # bytes per parameter

    # Step 1: Weight memory
    weight_gb = m['params_B'] * bpp  # billions of params * bytes = GB
    print(f'Step 1: Weight Memory')  # Display output
    print(f'  params x bytes_per_param = {m["params_B"]}B x {bpp} = {weight_gb:.1f} GB')  # Display output
    print()  # Display output

    # Step 2: KV cache per token (bytes, then convert)
    # Formula: 2 (K+V) x heads x head_dim x layers x bytes_per_value
    kv_bytes_per_token = 2 * m.get('kv_heads', m['heads']) * m['head_dim'] * m['layers'] * bpp  # Assign value
    kv_mb_per_token = kv_bytes_per_token / (1024**2)  # Assign value
    print(f'Step 2: KV Cache Per Token')  # Display output
    print(f'  2 (K+V) x {m["heads"]} heads x {m["head_dim"]} dim x {m["layers"]} layers x {bpp} bytes')  # Display output
    print(f'  = {kv_bytes_per_token:,.0f} bytes = {kv_mb_per_token:.2f} MB/token')  # Display output
    print()  # Display output

    # Step 3: KV cache per user
    kv_per_user_gb = (kv_bytes_per_token * tokens_per_conversation) / (1024**3)  # Assign value
    print(f'Step 3: KV Cache Per User')  # Display output
    print(f'  {kv_mb_per_token:.2f} MB/token x {tokens_per_conversation} tokens = {kv_per_user_gb:.2f} GB/user')  # Display output
    print()  # Display output

    # Step 4: Total KV cache for all users
    total_kv_gb = kv_per_user_gb * num_users  # Assign value
    print(f'Step 4: Total KV Cache')  # Display output
    print(f'  {kv_per_user_gb:.2f} GB/user x {num_users} users = {total_kv_gb:.1f} GB')  # Display output
    print()  # Display output

    # Step 5: Total memory (weights + KV + 10% overhead)
    overhead_gb = weight_gb * 0.10  # 10% of weights for activations/framework
    total_gb = weight_gb + total_kv_gb + overhead_gb  # Assign value
    print(f'Step 5: Total Memory Needed')  # Display output
    print(f'  weights + total_kv + overhead(10%) = {weight_gb:.1f} + {total_kv_gb:.1f} + {overhead_gb:.1f} = {total_gb:.1f} GB')  # Display output
    print()  # Display output

    # Step 6: Fit check
    fits = total_gb <= gpu_vram  # Assign value
    headroom = gpu_vram - total_gb  # Assign value
    print(f'Step 6: Does it fit?')  # Display output
    print(f'  GPU VRAM = {gpu_vram} GB')  # Display output
    print(f'  Needed   = {total_gb:.1f} GB')  # Display output
    if fits:  # Condition check
        print(f'  \U00002705 FITS ({headroom:.1f} GB headroom)')  # Display output
    else:
        print(f'  \U0000274C OOM (need {-headroom:.1f} more GB)')  # Display output
    print()  # Display output

    # Stacked bar chart: weights | KV | overhead vs GPU VRAM
    fig_2, ax_2 = plt.subplots(figsize=(8, 2))  # Matplotlib setup
    ax_2.barh(['Memory'], [weight_gb], color='#dbeafe', edgecolor='#000', label='Weights')
    ax_2.barh(['Memory'], [total_kv_gb], left=[weight_gb], color='#fef3c7', edgecolor='#000', label='KV Cache')
    ax_2.barh(['Memory'], [overhead_gb], left=[weight_gb + total_kv_gb], color='#f3e8ff', edgecolor='#000', label='Overhead')
    # GPU VRAM line
    ax_2.axvline(gpu_vram, color='red', linestyle='--', linewidth=2, label=f'{gpu_name} VRAM ({gpu_vram} GB)')  # Add vertical line
    ax_2.set_xlabel('GB')  # Set axis label
    ax_2.set_title('Memory Breakdown vs GPU Capacity')  # Set title
    ax_2.legend(loc='upper right', fontsize=8)  # Add legend
    ax_2.set_xlim(0, max(total_gb, gpu_vram) * 1.1)  # Set axis limits
    plt.tight_layout()  # Adjust layout
    plt.show()  # Render figure

# Build interactive widgets
widgets.interact(  # Widget element
    capacity_calc,
    model_name=widgets.Dropdown(options=list(MODELS.keys()), value='Mistral-7B', description='Model:'),  # Widget element
    gpu_name=widgets.Dropdown(options=list(GPUS.keys()), value='A100-80', description='GPU:'),  # Widget element
    precision=widgets.Dropdown(options=list(PRECISIONS.keys()), value='FP16', description='Precision:'),  # Widget element
    tokens_per_conversation=widgets.IntSlider(min=256, max=32768, step=256, value=2048, description='Tokens/conv:',  # Widget element
                                              style={'description_width': 'initial'}),  # Assign value
    num_users=widgets.IntSlider(min=1, max=256, step=1, value=16, description='Concurrent users:',  # Widget element
                                style={'description_width': 'initial'}),  # Assign value
);

interactive(children=(Dropdown(description='Model:', options=('Mistral-7B', 'Llama-70B', 'Llama-405B'), value=…

### Interactive Instance Selector

## Use-Case Driven Capacity Planner

**The core tension:** on a fixed model, three things compete for the same GPU — **latency**, **context** (quality), and **throughput** (batch). You cannot maximize all three. So don\'t try.

**The framework:** pick a use case. The use case decides which dimension you **pin** (hard constraint), which you **chase** (the goal), and which becomes the **residual** (whatever\'s left over). The planner then sweeps the GPU catalog and crowns the winner — the cheapest GPU that satisfies the constraint while maximizing the goal.

| Use case | PIN (constraint) | CHASE (goal) | SOLVE FOR (residual) |
|---|---|---|---|
| **Premium chat** | Latency ≤ target | Context / quality | → max batch that fits |
| **Async agents** | Context (long) | Throughput (batch) | → resulting latency |
| **Batch / offline** | (latency free) | Throughput + $/M tok | → throughput ceiling |

**The payoff:** same model, three use cases, *three different winning GPUs*. The right GPU for chat is the wrong GPU for agents. That\'s the whole story.

In [4]:
# === Capacity Planner: fix SLO + min batch, slide context, rank surviving GPUs ===

ARCH = {
    "7B":   {"kv_heads": 8, "head_dim": 128, "layers": 32},
    "13B":  {"kv_heads": 8, "head_dim": 128, "layers": 40},
    "70B":  {"kv_heads": 8, "head_dim": 128, "layers": 80},
    "405B": {"kv_heads": 8, "head_dim": 128, "layers": 126},
}

def kv_per_token_bytes(arch):
    # 2 (K+V) x kv_heads x head_dim x layers x 2 bytes (KV in FP16)
    return 2 * arch["kv_heads"] * arch["head_dim"] * arch["layers"] * 2

def evaluate(model, precision, batch, context):
    arch = ARCH[model]
    model_bytes = MODEL_PARAMS[model] * PRECISION_BYTES[precision] * 1e9   # weights in bytes
    kv_per_user = kv_per_token_bytes(arch) * context                      # bytes per user (full context)
    rows = []
    for name, spec in GPU_INSTANCES.items():
        bw = spec["bandwidth_gbs"] * 1e9                                   # bytes/sec
        # ITL: stream weights + every user's KV once per decode step
        itl_ms = (model_bytes + batch * kv_per_user) / bw * 1000.0
        throughput = batch * (1000.0 / itl_ms)                            # tokens/sec
        cost_m = (spec["cost_per_hr"] / 3600.0) / throughput * 1e6 if throughput > 0 else float("inf")
        # memory fit: weights + all users' KV
        total_vram = (model_bytes + batch * kv_per_user) / 1e9
        fits = total_vram <= spec["vram_gb"] * 0.9
        rows.append({"name": name, "itl": itl_ms, "tput": throughput,
                     "cost_m": cost_m, "vram": total_vram, "fits": fits})
    return rows

def render(_=None):
    out.clear_output()
    with out:
        slo = slo_sl.value
        batch = batch_sl.value
        rows = evaluate(model_dd.value, prec_dd.value, batch, ctx_sl.value)
        options = [r for r in rows if r["fits"] and r["itl"] < slo]
        options.sort(key=lambda r: r["cost_m"])
        dropped = [r for r in rows if not (r["fits"] and r["itl"] < slo)]

        display(HTML(
            f"<div style='font:600 13px sans-serif;color:#374151;margin:2px 0 8px'>"
            f"{model_dd.value} {prec_dd.value} &nbsp;|&nbsp; SLO &lt; {slo}ms "
            f"&nbsp;|&nbsp; min batch {batch} &nbsp;|&nbsp; context {ctx_sl.value}</div>"))

        if options:
            w = options[0]
            display(HTML(
                f"<div style='background:#dcfce7;border:1px solid #16a34a;border-radius:8px;"
                f"padding:10px;font:14px sans-serif;margin-bottom:8px'>"
                f"🏆 <b>Cheapest option: {w['name']}</b> &nbsp; "
                f"ITL <b>{w['itl']:.0f}ms</b> &nbsp;|&nbsp; <b>{w['tput']:.0f}</b> tok/s "
                f"&nbsp;|&nbsp; <b>${w['cost_m']:.2f}</b>/M tok &nbsp;|&nbsp; "
                f"{len(options)} option(s)</div>"))
        else:
            display(HTML("<div style='color:#b91c1c;font:14px sans-serif'>"
                         "❌ No GPU holds the SLO at this batch + context. "
                         "Raise SLO, lower batch, or shorten context.</div>"))

        html = "<table style='border-collapse:collapse;width:100%;font:13px sans-serif'>"
        html += ("<tr style='background:#f3f4f6'><th align='left'>GPU</th><th>$/hr</th><th>ITL</th>"
                 "<th>tok/s</th><th>$/M tok</th><th>VRAM</th><th>Status</th></tr>")
        for r in options:
            bg = "#dcfce7" if r is options[0] else "#ffffff"
            html += (f"<tr style='background:{bg}'><td>{r['name']}</td>"
                     f"<td align='center'>${GPU_INSTANCES[r['name']]['cost_per_hr']:.2f}</td>"
                     f"<td align='center'>{r['itl']:.0f}ms</td>"
                     f"<td align='center'>{r['tput']:.0f}</td>"
                     f"<td align='center'>${r['cost_m']:.2f}</td>"
                     f"<td align='center'>{r['vram']:.0f}/{GPU_INSTANCES[r['name']]['vram_gb']}GB</td>"
                     f"<td align='center'>✅</td></tr>")
        for r in dropped:
            why = "won\'t fit" if not r["fits"] else f"ITL {r['itl']:.0f}ms &gt; SLO"
            html += (f"<tr style='background:#ffe4e6;color:#9f1239'><td>{r['name']}</td>"
                     f"<td align='center'>${GPU_INSTANCES[r['name']]['cost_per_hr']:.2f}</td>"
                     f"<td align='center'>{r['itl']:.0f}ms</td>"
                     f"<td align='center'>{r['tput']:.0f}</td><td align='center'>—</td>"
                     f"<td align='center'>{r['vram']:.0f}/{GPU_INSTANCES[r['name']]['vram_gb']}GB</td>"
                     f"<td align='center'>{why}</td></tr>")
        html += "</table>"
        display(HTML(html))

# --- requirements you FIX, and context you SLIDE ---
model_dd = widgets.Dropdown(options=list(MODEL_PARAMS.keys()), value="7B", description="Model:")
prec_dd  = widgets.Dropdown(options=list(PRECISION_BYTES.keys()), value="FP16", description="Precision:")
slo_sl   = widgets.IntSlider(value=50, min=5, max=200, step=5, description="Max SLO ms:")
batch_sl = widgets.IntSlider(value=16, min=1, max=256, step=1, description="Min batch:")
ctx_sl   = widgets.IntSlider(value=4096, min=512, max=32768, step=512, description="Context:")
out      = widgets.Output()

for w in [model_dd, prec_dd, slo_sl, batch_sl, ctx_sl]:
    w.observe(render, names="value")

display(widgets.VBox([model_dd, prec_dd, slo_sl, batch_sl, ctx_sl, out]))
render()

## Summary

The capacity equation:
```
Max users = (GPU_VRAM - model_weights - overhead) / (KV_per_token x context_length)
```

What determines your cost:
- **Model precision** (FP16 vs INT4) sets weight memory
- **Context length** sets KV per user
- **SLO** caps batch size, which caps throughput
- **GPU choice** (VRAM + bandwidth) determines $/M tokens

**Next:** How to push these limits with optimizations (GQA, prefix caching, engines).